# In Class Assignment 2026.04.07

## Imports

In [49]:
import pandas as pd
import numpy as np
import sweetviz as sv

In [50]:
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn import tree
from sklearn.base import clone
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, precision_recall_fscore_support, RocCurveDisplay, roc_auc_score, cohen_kappa_score, r2_score, mean_squared_error
import matplotlib.pyplot as plt

## (3 points) Using pandas, load the dataset and display the first 5 rows.

In [51]:
df = pd.read_csv("insurance.csv")

In [52]:
df.head(5)

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## (3 points) Using SweetViz, do a quick EDA of the data set. In a markdown cell describe the main trends and patterns you see in the data.

In [53]:
report = sv.analyze(df)
report.show_html("insurance_report.html")

                                             |          | [  0%]   00:00 -> (? left)

Report insurance_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


It's interesting that `charges` and `smoker` are so highly correlated.

There is good balance of sex, and surprisingly, 20% of the observations are smokers.

The data seems to be quite well-distributed, with every input variable having a mature distribution, meaning that our models should have the ability to work well for test data.

## (4 points) Define the input variables (X) and target variable (y). Split the data into training (80%) and testing (20%). 

In [54]:
df_enc = pd.get_dummies(df, drop_first=True).astype(int)

In [55]:
df_enc.head(5)

,age,bmi,children,charges,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27,0,16884,0,1,0,0,1
1,18,33,1,1725,1,0,0,1,0
2,28,33,3,4449,1,0,0,1,0
3,33,22,0,21984,1,0,1,0,0
4,32,28,0,3866,1,0,1,0,0


In [60]:
X = df_enc.drop(columns=["charges"])
y = df_enc["charges"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## (4 points) Scale the input variables.

In [61]:
X_train_scaled = MinMaxScaler().fit_transform(X_train.select_dtypes(include=np.number))

## (5 points) Build a baseline linear regressor and random forest regressor. Report both R2 and MSE for both models. Add a markdown cell discussing the performance of each of the models on the test data.

In [62]:
baseline_lr = LinearRegression()
baseline_lr.fit(X_train_scaled, y_train)
y_pred_lr = baseline_lr.predict(X_test_scaled := MinMaxScaler().fit_transform(X_test.select_dtypes(include=np.number)))
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)
print(f"Baseline Linear Regression MSE: {mse_lr:.2f}, R2: {r2_lr:.2f}")

Baseline Linear Regression MSE: 33592199.36, R2: 0.78


In [63]:
baseline_rf = RandomForestRegressor(n_estimators=100, random_state=42)
baseline_rf.fit(X_train, y_train)
y_pred_rf = baseline_rf.predict(X_test)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)
print(f"Baseline Random Forest MSE: {mse_rf:.2f}, R2: {r2_rf:.2f}")

Baseline Random Forest MSE: 22179121.67, R2: 0.86


The linear model explains 78% of the variation of the `charges` variable, while the random forest model explains 87% of the variation of `charges`.  The random forest model has a lower MSE than the linear regressor.  As such, the random forest performed better on the test data than the linear model.

## (5 points) Use GridSearchCV to tune n_estimators, max_depth, min_samples_split and max_features. What is the best set of hyperparameters from your search?

In [64]:
cv_lr = cross_val_score(baseline_lr, X_train_scaled, y_train, cv=5, scoring="r2")
print(f"Baseline Linear Regression CV R2 Scores: {cv_lr}")

Baseline Linear Regression CV R2 Scores: [0.71508701 0.8018118  0.72333011 0.65748491 0.76763199]


In [65]:
cv_rf = cross_val_score(baseline_rf, X_train, y_train, cv=5, scoring="r2")
print(f"Baseline Random Forest CV R2 Scores: {cv_rf}")

Baseline Random Forest CV R2 Scores: [0.81530553 0.89851976 0.79006114 0.78263259 0.82825938]


In [66]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "max_features": ["sqrt", "log2"]
}
gscv_rf = GridSearchCV(estimator=RandomForestRegressor(random_state=42), param_grid=param_grid, cv=5, n_jobs=-1)
gscv_rf.fit(X_train, y_train)
print(f"Best RF Parameters: {gscv_rf.best_params_}")
print(f"Best RF CV Score: {gscv_rf.best_score_:.4f}")

Best RF Parameters: {'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 200}
Best RF CV Score: 0.8412


## (5 points) Examine the top 20 models (based on rank) from your GridSearchCV results. Add a markdown cell and discuss which set of hyperparameters you would choose and why. 

In [67]:
cv_results_rf = pd.DataFrame(gscv_rf.cv_results_)
cv_results_rf = cv_results_rf.sort_values(by="mean_test_score", ascending=False).head(20)
cv_results_rf

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_max_depth,param_max_features,param_min_samples_split,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
15,0.563815,0.024125,0.026113,0.002563,10,log2,5,200,"{'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 200}",0.837799,0.913115,0.821963,0.789291,0.843840,0.841202,0.040632,1
7,0.643873,0.021435,0.023624,0.001378,None,log2,5,200,"{'max_depth': None, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 200}",0.838805,0.909897,0.822971,0.788150,0.843484,0.840662,0.039687,2
23,0.425680,0.018167,0.015809,0.001567,20,log2,5,200,"{'max_depth': 20, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 200}",0.838805,0.909897,0.822971,0.788150,0.843484,0.840662,0.039687,2
14,0.280757,0.016878,0.013107,0.001628,10,log2,5,100,"{'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 100}",0.836136,0.912301,0.821322,0.789734,0.842765,0.840452,0.040310,4
12,0.330628,0.021008,0.017628,0.001663,10,log2,2,100,"{'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 2, 'n_estimators': 100}",0.836086,0.914660,0.818188,0.788628,0.844351,0.840383,0.041767,5
13,0.584254,0.017109,0.030378,0.006841,10,log2,2,200,"{'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 2, 'n_estimators': 200}",0.838204,0.913784,0.817449,0.787288,0.843111,0.839967,0.041819,6
22,0.271260,0.020717,0.012050,0.001850,20,log2,5,100,"{'max_depth': 20, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 100}",0.837289,0.909139,0.820870,0.784778,0.843900,0.839195,0.040534,7
6,0.293091,0.036094,0.014514,0.001799,None,log2,5,100,"{'max_depth': None, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 100}",0.837289,0.909139,0.820870,0.784778,0.843900,0.839195,0.040534,7
21,0.609828,0.026793,0.022317,0.002033,20,log2,2,200,"{'max_depth': 20, 'max_features': 'log2', 'min_samples_split': 2, 'n_estimators': 200}",0.835442,0.909107,0.812477,0.787410,0.838761,0.836639,0.040657,9
5,0.738339,0.044982,0.033765,0.002893,None,log2,2,200,"{'max_depth': None, 'max_features': 'log2', 'min_samples_split': 2, 'n_estimators': 200}",0.835371,0.909010,0.812723,0.787320,0.838540,0.836593,0.040613,10


In [68]:
pd.set_option('display.max_colwidth', None)
cv_results_rf["params"]

15      {'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 200}
7     {'max_depth': None, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 200}
23      {'max_depth': 20, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 200}
14      {'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 100}
12      {'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 2, 'n_estimators': 100}
13      {'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 2, 'n_estimators': 200}
22      {'max_depth': 20, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 100}
6     {'max_depth': None, 'max_features': 'log2', 'min_samples_split': 5, 'n_estimators': 100}
21      {'max_depth': 20, 'max_features': 'log2', 'min_samples_split': 2, 'n_estimators': 200}
5     {'max_depth': None, 'max_features': 'log2', 'min_samples_split': 2, 'n_estimators': 200}
20      {'max_depth': 20, 'max_features': 'log2', 

The first 12 best models all use log2 max_features, so I would definitely choose `log2` features vs `sqrt` features.  

It is more unclear for `max_depth`, since the estimators don't consistently have a max depth of 20 at the top.  

It seems like 5 is a better value than 2 for `min_samples_split`.

And then, it seems like 200 is the better value for `n_estimators` than 100.

## (3 points--if time) Compare the training versus test performance using a Random Forest model tuned with the best parameters from GridSearch CV. Is your model overfitting and how do you know?

In [69]:
train_error = mean_squared_error(y_train, gscv_rf.predict(X_train))
test_error = mean_squared_error(y_test, gscv_rf.predict(X_test))
print(f"Train MSE: {train_error:.2f}, Test MSE: {test_error:.2f}")

Train MSE: 10067783.72, Test MSE: 20430805.91


In [70]:
print(f"Ratio of test to train MSE: {test_error / train_error:.2f}")

Ratio of test to train MSE: 2.03


The test MSE is 2.42 times larger then the train MSE.  This alone isn't evidence to conclude overfitting, especially if the data is noisy, but we would like the test-to-train MSE ratio to be smaller.